# SciEntsBank Baseline Grading Evaluation

Evaluate LLM grading accuracy on [SciEntsBank](https://huggingface.co/datasets/nkazi/SciEntsBank) using a simple baseline prompt.

- **Metric**: Quadratic Weighted Kappa (QWK), `weights="quadratic"`
- **Labels**: 0 = incorrect, 1 = partially correct, 2 = correct
- **Default model**: Qwen3-4B-Instruct

Change `ACTIVE_MODEL_KEY` in the config cell to swap models.

In [ ]:
# Cell 1: Install dependencies
!pip install -q datasets transformers accelerate scikit-learn tqdm

In [ ]:
# Cell 2: Configuration — swap models here
from dataclasses import dataclass
from typing import Optional

# ── Model registry (add new models here) ──────────────────────────────
MODEL_REGISTRY = {
    "qwen3-4b":      "Qwen/Qwen3-4B-Instruct-2507",
    "qwen2.5-3b":    "Qwen/Qwen2.5-3B-Instruct",
    "mistral-7b":    "mistralai/Mistral-7B-Instruct-v0.3",
}

ACTIVE_MODEL_KEY = "qwen3-4b"  # <-- change this to switch models

@dataclass
class EvalConfig:
    model_key: str = ACTIVE_MODEL_KEY
    eval_split: str = "test_ua"          # train | test_ua | test_uq | test_ud
    max_samples: Optional[int] = None    # None = full split; set e.g. 50 for quick test
    max_new_tokens: int = 8
    temperature: float = 0.0
    seed: int = 42

cfg = EvalConfig()

assert cfg.model_key in MODEL_REGISTRY, f"Unknown model key: {cfg.model_key}"
MODEL_ID = MODEL_REGISTRY[cfg.model_key]

print(f"Active model : {cfg.model_key} -> {MODEL_ID}")
print(f"Eval split   : {cfg.eval_split}")
print(f"Max samples  : {cfg.max_samples or 'all'}")

Active model : qwen3-4b -> Qwen/Qwen3-4B-Instruct-2507
Eval split   : test_ua
Max samples  : all


In [ ]:
# Cell 3: Load SciEntsBank and map labels to 0 / 1 / 2
from datasets import load_dataset

# 5-way SciEntsBank label -> baseline grading label
#   2 = correct, 1 = partially correct, 0 = incorrect
LABEL_MAP_5WAY = {
    "correct": 2,
    "partially_correct_incomplete": 1,
    "contradictory": 0,
    "irrelevant": 0,
    "non_domain": 0,
}

dataset = load_dataset("nkazi/SciEntsBank")
eval_ds = dataset[cfg.eval_split]

if cfg.max_samples is not None:
    eval_ds = eval_ds.select(range(min(cfg.max_samples, len(eval_ds))))

def get_gold_label(example: dict) -> int:
    """Convert dataset label to integer 0/1/2."""
    label_name = example["label"]
    if isinstance(label_name, int):
        # ClassLabel int -> name lookup
        label_name = dataset[cfg.eval_split].features["label"].names[label_name]
    return LABEL_MAP_5WAY[label_name]

print(f"Loaded {len(eval_ds)} samples from split '{cfg.eval_split}'")
print(f"Columns: {eval_ds.column_names}")
print("\nExample:")
ex = eval_ds[0]
print(f"  question          : {ex['question'][:80]}...")
print(f"  reference_answer  : {ex['reference_answer']}")
print(f"  student_answer    : {ex['student_answer']}")
print(f"  gold label (0/1/2) : {get_gold_label(ex)}")

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

data/train-00001.parquet:   0%|          | 0.00/233k [00:00<?, ?B/s]

data/test-ua-00001.parquet:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

data/test-uq-00001.parquet:   0%|          | 0.00/35.7k [00:00<?, ?B/s]

data/test-ud-00001.parquet:   0%|          | 0.00/177k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4969 [00:00<?, ? examples/s]

Generating test_ua split:   0%|          | 0/540 [00:00<?, ? examples/s]

Generating test_uq split:   0%|          | 0/733 [00:00<?, ? examples/s]

Generating test_ud split:   0%|          | 0/4562 [00:00<?, ? examples/s]

Loaded 540 samples from split 'test_ua'
Columns: ['id', 'question', 'reference_answer', 'student_answer', 'label']

Example:
  question          : You used several methods to separate and identify the substances in mock rocks. ...
  reference_answer  : The water was evaporated, leaving the salt.
  student_answer    : We evaporated the water.
  gold label (0/1/2) : 2


In [ ]:
# Cell 4: Baseline grading prompt and response parser
import re

BASELINE_PROMPT = """You are a university professor for an introductory class.
Your job is to grade exercises and decide if the student answers are correct(2), partially correct(1), or incorrect(0).
Return the corresponding integer label for the grading, 0 for incorrect, 1 for partially correct, 2 for correct.

Question: {question}
Reference Answer: {reference_answer}
Student Answer: {student_answer}"""

def build_prompt(question: str, reference_answer: str, student_answer: str) -> str:
    return BASELINE_PROMPT.format(
        question=question,
        reference_answer=reference_answer,
        student_answer=student_answer,
    )

def parse_grade(response: str) -> int:
    """Extract integer label 0/1/2 from model output. Returns -1 on parse failure."""
    text = response.strip()
    if text in ("0", "1", "2"):
        return int(text)
    match = re.search(r"\b([012])\b", text)
    if match:
        return int(match.group(1))
    return -1

# Quick sanity check
assert parse_grade("2") == 2
assert parse_grade("The answer is 1.") == 1
assert parse_grade("0 - incorrect") == 0
print("Prompt template and parser ready.")

Prompt template and parser ready.


In [ ]:
# Cell 5: LLM interface — load model and run inference
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

class LLMGrader:
    """Unified grader interface; swap model by changing MODEL_ID / ACTIVE_MODEL_KEY."""

    def __init__(self, model_id: str, max_new_tokens: int = 8, temperature: float = 0.0):
        self.model_id = model_id
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
            trust_remote_code=True,
        )
        self.model.eval()

    def grade(self, question: str, reference_answer: str, student_answer: str) -> tuple[int, str]:
        """Return (parsed_label, raw_response)."""
        user_content = build_prompt(question, reference_answer, student_answer)
        messages = [{"role": "user", "content": user_content}]

        # Use chat template when available; fall back to plain prompt
        if hasattr(self.tokenizer, "apply_chat_template") and self.tokenizer.chat_template:
            prompt = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        else:
            prompt = user_content

        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=self.temperature > 0,
                temperature=self.temperature if self.temperature > 0 else None,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        raw = self.tokenizer.decode(new_tokens, skip_special_tokens=True)
        return parse_grade(raw), raw


grader = LLMGrader(MODEL_ID, max_new_tokens=cfg.max_new_tokens, temperature=cfg.temperature)
print(f"Model loaded: {MODEL_ID}")
print(f"Device: {grader.model.device}")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen3-4B-Instruct-2507
Device: cuda:0


In [ ]:
# Cell 6: Run evaluation — collect predictions vs gold labels
from tqdm.auto import tqdm

y_true, y_pred = [], []
records = []

for i, example in enumerate(tqdm(eval_ds, desc=f"Grading [{cfg.model_key}]")):
    gold = get_gold_label(example)
    pred, raw = grader.grade(
        example["question"],
        example["reference_answer"],
        example["student_answer"],
    )
    y_true.append(gold)
    y_pred.append(pred)
    records.append({
        "id": example["id"],
        "gold": gold,
        "pred": pred,
        "raw_response": raw,
        "question": example["question"],
        "reference_answer": example["reference_answer"],
        "student_answer": example["student_answer"],
    })

parse_failures = sum(1 for p in y_pred if p == -1)
print(f"\nEvaluated {len(y_true)} samples")
print(f"Parse failures: {parse_failures}")

Grading [qwen3-4b]:   0%|          | 0/540 [00:00<?, ?it/s]


Evaluated 540 samples
Parse failures: 10


In [ ]:
# Cell 7: Compute QWK (quadratic weighted kappa) and summary metrics
import pandas as pd
from sklearn.metrics import cohen_kappa_score, accuracy_score, classification_report

# Filter out parse failures for fair metric computation
valid_idx = [i for i, p in enumerate(y_pred) if p in (0, 1, 2)]
y_true_valid = [y_true[i] for i in valid_idx]
y_pred_valid = [y_pred[i] for i in valid_idx]

qwk = cohen_kappa_score(y_true_valid, y_pred_valid, weights="quadratic")
acc = accuracy_score(y_true_valid, y_pred_valid)

print("=" * 50)
print(f"Model        : {cfg.model_key} ({MODEL_ID})")
print(f"Split        : {cfg.eval_split}")
print(f"Samples      : {len(y_true_valid)} valid / {len(y_true)} total")
print(f"QWK          : {qwk:.4f}")
print(f"Accuracy     : {acc:.4f}")
print("=" * 50)

print("\nClassification report (labels: 0=incorrect, 1=partial, 2=correct):")
print(classification_report(
    y_true_valid, y_pred_valid,
    labels=[0, 1, 2],
    target_names=["incorrect (0)", "partial (1)", "correct (2)"],
    zero_division=0,
))

# Preview misclassified examples
df = pd.DataFrame(records)
df_valid = df[df["pred"].isin([0, 1, 2])].copy()
df_valid["correct"] = df_valid["gold"] == df_valid["pred"]
print(f"\nMisclassified examples ({(~df_valid['correct']).sum()}):")
display(df_valid[~df_valid["correct"]].head(10))

Model        : qwen3-4b (Qwen/Qwen3-4B-Instruct-2507)
Split        : test_ua
Samples      : 530 valid / 540 total
QWK          : 0.6741
Accuracy     : 0.6302

Classification report (labels: 0=incorrect, 1=partial, 2=correct):
               precision    recall  f1-score   support

incorrect (0)       0.79      0.73      0.76       194
  partial (1)       0.32      0.53      0.39       110
  correct (2)       0.80      0.60      0.69       226

     accuracy                           0.63       530
    macro avg       0.64      0.62      0.61       530
 weighted avg       0.70      0.63      0.65       530


Misclassified examples (196):


,id,gold,pred,raw_response,question,reference_answer,student_answer,correct
5,EM.45c.385.1,0,1,"1\n\nExplanation: The student mentions """,You used several methods to separate and ident...,The crystals were square with Xs on the surface.,By looking very closely at it.,False
6,EM.45c.396.1,0,1,1\n\nExplanation: The student mentions that,You used several methods to separate and ident...,The crystals were square with Xs on the surface.,We know the crystals were salt because we iden...,False
7,EM.45c.645.1,0,1,1\n\nExplanation: The student mentions looking,You used several methods to separate and ident...,The crystals were square with Xs on the surface.,Because I look at the chart.,False
8,EM.16b.439.1,2,0,0\n\nExplanation: The student's answer,"Ms. Teridann, a geologist, made a chart showin...",Topaz would be the hardest of the 4 minerals b...,Topaz is most like mineral Z because Topaz can...,False
10,EM.16b.593.1,1,2,2,"Ms. Teridann, a geologist, made a chart showin...",Topaz would be the hardest of the 4 minerals b...,"Topaz is harder than X, Y, Z.",False
11,EM.16b.645.1,0,1,1\n\nExplanation: The student's answer,"Ms. Teridann, a geologist, made a chart showin...",Topaz would be the hardest of the 4 minerals b...,Topaz would be the hardest because none of the...,False
12,EM.21a.403.1,1,2,2,Georgia found one brown mineral and one black ...,Rub the minerals together and see which one sc...,She can rub them against each other.,False
13,EM.21a.557.1,1,2,2,Georgia found one brown mineral and one black ...,Rub the minerals together and see which one sc...,Rubbing them together.,False
17,EM.21b.357.1,1,0,0\n\nExplanation: The student's answer,Georgia found one brown mineral and one black ...,The harder mineral will leave a scratch on the...,That it stay the scratch.,False
19,EM.21b.645.1,2,1,1\n\nExplanation: The student's answer,Georgia found one brown mineral and one black ...,The harder mineral will leave a scratch on the...,The harder one leave a scratch on the less scr...,False
